# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the field `@id`s and names in each record set.

In [ ]:
# List available record sets (`@id` and `name`)
print("Available record sets:")
for recset in dataset.record_sets:
    print(f"- @id: {recset.id}, name: {recset.name if hasattr(recset, 'name') else 'N/A'}")
    print("  Fields:")
    for field in recset.fields:
        field_name = getattr(field, 'name', 'N/A')
        print(f"    - @id: {field.id}, name: {field_name}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into pandas DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Show available record sets and the first few columns for each
for rid in dataframes:
    print(f"Record set @id: {rid}")
    print("Columns:", dataframes[rid].columns.tolist())
    display(dataframes[rid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data, or grouping data by key attributes.

In [ ]:
# Choose a record set for analysis
# Replace with the actual `@id` of the table you want to analyze (use info from section 2)
selected_record_set = None
for rid, df in dataframes.items():
    if not df.empty:
        selected_record_set = rid
        break

if selected_record_set is None:
    print("No non-empty record set found for EDA.")
else:
    df = dataframes[selected_record_set]
    print(f"Working with record set: {selected_record_set}")
    
    # Try to locate a numeric field for analysis
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if not numeric_fields:
        # Try to convert columns to numeric and see if any work
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
        if not numeric_fields:
            print("No numeric field available for EDA.")
    
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered rows with {numeric_field} > {threshold} (mean):")
        print(filtered_df.head())
        
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field if available
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No groupable (categorical) field found.")
    else:
        print("No numeric field found for normalization and filtering.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and not df.empty and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {selected_record_set}")
    plt.xlabel(numeric_field)
    plt.show()

    # If there's a group/categorical field, plot boxplot
    if group_fields:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used the `mlcroissant` library to load metadata and records from the FAIR^2 dataset, explored its record sets and fields, extracted tabular data into pandas DataFrames, performed basic EDA such as filtering and normalizing numeric fields, grouped by categorical attributes, and visualized distributions. For further research, consult the source Croissant schema and publication.